In [96]:
import torch
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

words = open('names.txt', 'r').read().splitlines()
print(words[:5])
print(len(words))

cuda
['emma', 'olivia', 'ava', 'isabella', 'sophia']
32033


In [27]:
# encode chars to integers
chars = sorted(list(set(''.join(words))))
stoi = { ch:i+1 for i,ch in enumerate(chars)}
stoi['.'] = 0
itos = { i:ch for ch,i in stoi.items()}
vocab_size = len(itos)
print(itos)

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello"))
print(decode([8, 9, 0]))
print(encode(words[0][0]))

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
[8, 5, 12, 12, 15]
hi.
[5]


In [ ]:
# build dataset
'''
[...] -> [b]
[..b] -> [o]
[.bo] -> [b]
[bob] -> [.]
'''
block_size = 4

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        # sliding window
        for ch in w + '.':
            ix = stoi[ch] # encode(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

n1 = int(0.9*len(words))

X, Y = build_dataset(words)
Xtr, Ytr = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:])

print(X[:5])
print(Y[:5])
len(Xtr)

tensor([[ 0,  0,  0,  0],
        [ 0,  0,  0,  5],
        [ 0,  0,  5, 13],
        [ 0,  5, 13, 13],
        [ 5, 13, 13,  1]])
tensor([ 5, 13, 13,  1,  0])


205411

In [ ]:
batch_size = 32

def get_batch(split):
    dataX = Xtr if split == 'train' else Xval
    dataY = Ytr if split == 'train' else Yval

    ix = torch.randint(low=0, high=dataX.shape[0], size=(batch_size, ))
    x = torch.stack([dataX[i] for i in ix])
    y = torch.stack([dataY[i] for i in ix])

    x, y = x.to(device), y.to(device)
    return x, y

# Xb, Yb = get_batch('train')
# print(decode(Xb[:1][0].tolist()))
# print(decode(Yb[:1].tolist()))


.mag
a


In [178]:
# create embedding -> FFN (tanh non-linearity)
emb_size = 64
n_hidden = 64
lr = 3e-3

C = torch.randn((vocab_size, emb_size), device=device)
W1 = torch.randn((block_size * emb_size, n_hidden), device=device)
B1 = torch.randn((n_hidden), device=device)
W2 = torch.randn((n_hidden, vocab_size), device=device)
B2 = torch.randn((vocab_size), device=device)
parameters = [C, W1, B1, W2, B2]
for p in parameters:
    p.requires_grad = True


In [ ]:
max_iters = 50000
for i in range(max_iters):
    Xb, Yb = get_batch('train')

    emb = C[Xb] # (B, T, C) -> (batch_size, block_size, emb_size)

    # concat (B, T*C) -> tanh linear
    x = emb.view(emb.shape[0], -1)
    x = torch.tanh(x @ W1 + B1)
    x = x @ W2 + B2

    # cross entropy loss (negative log likelihood)
    loss = F.cross_entropy(x, Yb)

    # zero grad
    for p in parameters:
        p.grad = None
    loss.backward()

    # # update
    for p in parameters:
        p.data += -lr * p.grad

    if i % 1000 == 0:
        print(loss.item())

2.030299663543701
3.3661699295043945
3.2722792625427246
3.337559461593628
3.0790843963623047
2.9050254821777344
2.5963544845581055
3.083397626876831
3.1728222370147705
2.6519980430603027
3.341979742050171
3.198549509048462
2.9127914905548096
2.7175042629241943
2.5772440433502197
4.20338773727417
2.4468886852264404
2.7790756225585938
2.6180546283721924
3.229918956756592
2.434370517730713
4.030163764953613
3.072611093521118
2.944790840148926
3.191521644592285
3.142058849334717
4.68748664855957
2.014059066772461
3.545607566833496
1.8983839750289917
2.0982718467712402
2.8322298526763916
4.75251579284668
2.9228405952453613
3.302654266357422
2.8186428546905518
2.116158962249756
2.8590643405914307
4.410735130310059
4.267761707305908
3.443666458129883
4.732165336608887
2.727166175842285
3.0991921424865723
4.136996746063232
4.309350490570068
4.0625786781311035
2.315325975418091
4.124395847320557
2.8862433433532715


In [ ]:
# AdamW update variant
optimizer = torch.optim.AdamW(parameters, lr=lr)
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()

In [215]:
torch.save(parameters, 'parameters.pt')

In [259]:
# generation
@torch.no_grad()
def generate():
    out = []
    context = [0] * block_size
    while True:
        Xgen = torch.tensor([context], device=device)
        emb = C[Xgen] # (B, T, C) -> (batch_size, block_size, emb_size)
        # concat (B, T*C) -> tanh linear
        x = emb.view(emb.shape[0], -1)
        x = torch.tanh(x @ W1 + B1)
        x = x @ W2 + B2
        probs = F.softmax(x, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(out)
    print(decode(out))

for _ in range(5):
    generate()

    


[18, 14, 5, 1, 14, 1, 0]
rneana.
[1, 19, 1, 18, 15, 14, 0]
asaron.
[5, 12, 9, 19, 1, 8, 0]
elisah.
[25, 12, 5, 5, 0]
ylee.
[9, 13, 5, 14, 19, 1, 8, 0]
imensah.
